# YOLOv8m — VisDrone (drone person/car detector)

Trains **yolov8m** on VisDrone on a free Colab **T4 GPU** (~1-2 h vs ~4 days on CPU).

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`.

Everything (checkpoints + exports) is saved to **Google Drive** at `MyDrive/drone/runs/visdrone_m`, so a Colab disconnect doesn't lose progress — just re-run the cells and it resumes from `last.pt`.

Keep this tab open while it trains.

In [ ]:
# 1. Confirm we actually got a GPU (should list a Tesla T4)
!nvidia-smi

In [ ]:
# 2. Install Ultralytics (pulls a CUDA torch build automatically on Colab)
!pip -q install ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 3. Mount Google Drive so weights persist across sessions
from google.colab import drive
drive.mount('/content/drive')
import os
RUNS = '/content/drive/MyDrive/drone/runs'
os.makedirs(RUNS, exist_ok=True)
print('runs dir:', RUNS)

In [ ]:
# 4. Train (auto-downloads VisDrone ~2 GB the first time).
#    If a previous session left a last.pt on Drive, resume from it instead of restarting.
from ultralytics import YOLO
from pathlib import Path

last = Path(RUNS) / 'visdrone_m' / 'weights' / 'last.pt'
if last.exists():
    print('resuming from', last)
    model = YOLO(str(last))
    model.train(resume=True)
else:
    print('fresh start from yolov8m.pt')
    model = YOLO('yolov8m.pt')
    model.train(
        data='VisDrone.yaml', epochs=50, imgsz=640, batch=16,  # T4 fits batch=16 easily; try 32 for speed
        device=0, project=RUNS, name='visdrone_m', exist_ok=True,
    )

In [ ]:
# 5. Export the best weights for deployment: CoreML (Mac Neural Engine) + ONNX (Pi / laptop).
best = f'{RUNS}/visdrone_m/weights/best.pt'
m = YOLO(best)
print('classes:', m.names)
m.export(format='coreml', nms=True, imgsz=640)   # -> best.mlpackage (next to best.pt on Drive)
m.export(format='onnx',   nms=True, imgsz=640, opset=12)  # -> best.onnx
print('done. Files are in', f'{RUNS}/visdrone_m/weights/')

## Outputs
On your Google Drive under `MyDrive/drone/runs/visdrone_m/weights/`:
- `best.pt` — PyTorch weights
- `best.mlpackage` — CoreML for the Mac's Neural Engine
- `best.onnx` — ONNX for the laptop/Pi follower

Download those, or tell the assistant they're on Drive and it'll wire `best.onnx`/`best.mlpackage` into the follower the same way as the nano model.